In [0]:
%pip install seaborn

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor

# Cargar datos
df = pd.read_csv('/Workspace/Users/a01567228@tec.mx/cement-strength-prediction/datathon_data_gcc.csv')

# Limpieza rápida para el análisis (similar a Gold)
df = df.dropna(subset=["Strength1", "Strength7", "Strength28"])
# Seleccionar solo numéricas
num_df = df.select_dtypes(include=[np.number]).fillna(df.select_dtypes(include=[np.number]).median())

features = [c for c in num_df.columns if 'Strength' not in c]
targets = ["Strength1", "Strength7", "Strength28"]

# 1. Analisis de Correlación
correlations = {}
for t in targets:
    correlations[t] = num_df[features].corrwith(num_df[t])
    
corr_df = pd.DataFrame(correlations)

# 2. Análisis de Feature Importance (Random Forest rápido)
rf_importances = {}
for t in targets:
    rf = RandomForestRegressor(n_estimators=50, random_state=42)
    rf.fit(num_df[features], num_df[t])
    rf_importances[t] = rf.feature_importances_
    
imp_df = pd.DataFrame(rf_importances, index=features)

# Imprimir las correlaciones para ver el cambio
print("--- Evolución de la Correlación con el Target ---")
print(corr_df)

print("\n--- Evolución de Feature Importance (RF) ---")
print(imp_df)

# Graficar para visualizar
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

corr_df.plot(kind='bar', ax=axes[0], colormap='viridis')
axes[0].set_title('Evolución de Correlaciones (1 vs 7 vs 28 días)')
axes[0].set_ylabel('Correlación de Pearson')
axes[0].axhline(0, color='black', linewidth=0.8)

imp_df.plot(kind='bar', ax=axes[1], colormap='plasma')
axes[1].set_title('Evolución de Feature Importance (Random Forest)')
axes[1].set_ylabel('Importancia Relativa')

plt.tight_layout()
plt.savefig('feature_evolution.png')
plt.show()